In [0]:
%sql
CREATE TABLE IF NOT EXISTS project.silver.products_dim(
  product_sk BIGINT GENERATED ALWAYS AS IDENTITY,
  product_id STRING,
  product_name STRING,
  product_category STRING,
  product_brand STRING,
  product_price DECIMAL(10,2),
  ingested_timestamp TIMESTAMP,
  valid_from TIMESTAMP,
  valid_to TIMESTAMP,
  active_flag BOOLEAN
);

In [0]:
source_table = "project.silver.products_stage"
target_table = "project.silver.products_dim"
checkpoint_path = "/Volumes/project/_ops/streaming/products/"

In [0]:
from pyspark.sql.functions import col


In [0]:
def merge_products_dim(microBatchDF, batchId: int):
    if microBatchDF.isEmpty():
        return

    # Only keep *post-image* rows (i.e., the latest values for inserts/updates)
    cdf_filtered = microBatchDF.filter(
        col("_change_type").isin("insert", "update_postimage")
    )

    cdf_filtered.createOrReplaceTempView("products_stage_cdf")

    microBatchDF.sparkSession.sql(f"""
      MERGE INTO {target_table} AS t
      USING (
        SELECT
          product_id,
          product_name,
          category,
          brand,
          price,
          ingesttime,
          __START_AT AS valid_from,
          __END_AT   AS valid_to,
          (__END_AT IS NULL) AS active_flag
        FROM products_stage_cdf
      ) AS s
      ON  t.product_id = s.product_id
      AND t.valid_from = s.valid_from
      WHEN MATCHED THEN
        UPDATE SET
          t.valid_to           = s.valid_to,
          t.active_flag        = s.active_flag,
          t.ingested_timestamp = s.ingesttime
      WHEN NOT MATCHED THEN
        INSERT (
          product_id,
          product_name,
          product_category,
          product_brand,
          product_price,
          ingested_timestamp,
          valid_from,
          valid_to,
          active_flag
        )
        VALUES (
          s.product_id,
          s.product_name,
          s.category,
          s.brand,
          s.price,
          s.ingesttime,
          s.valid_from,
          s.valid_to,
          s.active_flag
        )
    """)

In [0]:
# Read only change data from the stage table
stg_cdf = (
    spark.readStream
        .format("delta")
        .option("readChangeFeed", "true")
        .option("startingVersion", 2)
        .table(source_table)
)

query = (
    stg_cdf.writeStream
        .foreachBatch(merge_products_dim)
        .option("checkpointLocation", checkpoint_path)
        .trigger(availableNow=True)
        .start()
)

In [0]:
%sql
select * from project.silver.products_dim


In [0]:
# %sql
# DESCRIBE DETAIL project.silver.products_stage
# -- 

In [0]:
%sql
DESCRIBE HISTORY project.silver.products_stage;

In [0]:
# %sql
# SELECT * FROM table_changes('project.silver.products_stage',3)

In [0]:
# %sql
# CREATE TABLE IF NOT EXISTS project.silver.products_dim_checkpoint (
#   last_version BIGINT
# );


In [0]:
# %sql
# MERGE INTO project.silver.products_dim AS t
# USING (
#   SELECT
#     product_id,
#     product_name,
#     category,
#     brand,
#     price,
#     ingesttime,
#     __START_AT AS valid_from,
#     __END_AT   AS valid_to,
#     (__END_AT IS NULL) AS is_curretent
#   FROM project.silver.products_stage
# ) AS s
# ON  t.product_id = s.product_id
# AND t.valid_from = s.valid_from
# WHEN MATCHED THEN
#   -- Only "close" or "flip" the existing version row
#   UPDATE SET
#     t.valid_to           = s.valid_to,
#     t.is_curretent       = s.is_curretent,
#     t.ingested_timestamp = s.ingesttime
# WHEN NOT MATCHED THEN
#   INSERT (
#     product_id,
#     product_name,
#     product_category,
#     product_brand,
#     product_price,
#     ingested_timestamp,
#     valid_from,
#     valid_to,
#     is_curretent
#   )
#   VALUES (
#     s.product_id,
#     s.product_name,
#     s.category,
#     s.brand,
#     s.price,
#     s.ingesttime,
#     s.valid_from,
#     s.valid_to,
#     s.is_curretent
#   );


In [0]:
# %sql
# select * from project.silver.products_dim order by product_sk

In [0]:
# /Workspace/Users/manikanthavivek@gmail.com/databricks_dlt_pyspark/src/project/Customers/Customer_Autoloader.ipynb